# Plots box training data and literature (figure m02 and m04)

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
# Directories
dir_zen = '../paper_ecosystem_carbon/output/zenodo/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'

---

### Data preparation

In [ ]:
# Get training data
df_train = pd.read_csv(f'{dir_zen}ecosystem_carbon_data_m02_m04_training.csv',
                       skiprows=18)

# Get literature data
df_lit = pd.read_csv(f'{dir_zen}ecosystem_carbon_data_m02_m04_literature.csv',
                       skiprows=4)

In [ ]:
# Define colors
# Seaborn tab10 palette
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', 
          '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

dict_colors = dict(houghton_2023 = colors[1], 
                   goldstein_2020=colors[2],
                   mackey_2020=colors[3], 
                   ipcc_2019=colors[4],
                   hansis_2015=colors[5])

# Define Markers
dict_marker = dict(Tropical='v', 
                   Subtropical='^', 
                   Temperate='>', 
                   Boreal='<', 
                   Temperate_Boreal = 'o', 
                   Global='o')

# Lists of tropical/subtropical and temperate/boreal/other ecosystems
list_rf_trop = ['trop_ef', 'trop_df']
list_rf_temp = ['temp_enf', 'temp_ebf', 'temp_dnf', 'temp_dbf',
                'glob_shrub', 'glob_tundr', 'glob_grass', 'glob_mount']

# Dictionary with for ecosystems
d_rf_labels = dict(trop_ef = 'Evergreen forest',
                   trop_df = 'Deciduous forest',
                   temp_enf = 'Evergreen nl. forest',
                   temp_ebf = 'Evergreen bl. forest',
                   temp_dnf = 'Deciduous nl. forest',
                   temp_dbf = 'Deciduous bl. forest',
                   glob_mount = 'Montane',
                   glob_shrub = 'Shrubland',
                   glob_tundr = 'Tundra',
                   glob_grass = 'Grassland')

In [ ]:
# Prepare literature data for plotting
# Assign columns needed for plotting
df_lit = df_lit.assign(
    publication_author = df_lit.publication.str.split().str.get(0).str.lower(),
    publication_year = df_lit.publication.str.split().str.get(-1),
    assigned_name2 = df_lit.assigned_name.str.replace(' ', '_').str.lower())


df_lit = df_lit.assign(
    publication_id = df_lit.publication_author + '_' + df_lit.publication_year, 
    publication_id2 = df_lit.publication_author + '_' + df_lit.publication_year + 
        '_' + df_lit.assigned_name2)

# Change value columns to dtype float
df_lit = df_lit.astype(dict(value_best=float, value_min=float, value_max=float))

# Replace zeros with nans
# Add column with maximum value
df_lit = df_lit.assign(
    value_best = df_lit.value_best.replace(0, np.nan),
    value_min = df_lit.value_min.replace(0, np.nan),
    value_max = df_lit.value_max.replace(0, np.nan),
    total_max = df_lit[['value_best', 'value_min', 'value_max']].max(axis=1)
    )

# Drop rows without value
df_lit = df_lit[~np.isnan(df_lit.total_max)].reset_index(drop=True)

# Assign colors
df_lit['color'] = [dict_colors[i] for i in df_lit.publication_id]

# Assign markers
df_lit['marker'] = [dict_marker[i] for i in df_lit.assigned_eco_zone]

---

### Functions

In [ ]:
def get_plot_settings(trop_temp):
    """Return ecosystem types, y-positions, and literature zones."""

    if trop_temp == "trop":
        return {
            "rf_pos": [-1, 0],
            "rf_etype": ["trop_ef", "trop_df",],
            "eco_zones": ["Tropical", "Subtropical",],
        }

    elif trop_temp == "temp":
        return {
            "rf_pos": [-7, -6, -5, -4, -3, -2, -1, 0],
            "rf_etype": [
                "temp_enf",
                "temp_ebf",
                "temp_dnf",
                "temp_dbf",
                "glob_shrub",
                "glob_tundr",
                "glob_grass",
                "glob_mount",
            ],
            "eco_zones": ["Temperate", "Boreal", "Temperate_Boreal", "Global",],
        }

    else:
        raise ValueError("trop_temp must be 'trop' or 'temp'")

In [ ]:
def sort_rf_etypes(ctype, rf_etypes):
    """Sort training ecosystem types by decreasing median."""

    return sorted(
        rf_etypes,
        key=lambda etype: df_train[f"{ctype}_{etype}"].median(),
        reverse=True,
    )

In [ ]:
def add_boxplot(ax, data, position, edge_color="#000000", 
                fill_color="#FFFFFF",):
    """Add one horizontal boxplot."""

    data = data.dropna()

    if data.empty:
        return

    bp = ax.boxplot(
        data,
        positions=[position],
        vert=False,
        widths=0.5,
        patch_artist=True,
        flierprops={
            "marker": ".",
            "alpha": 0.5,
        },
    )

    for element in [
        "boxes",
        "whiskers",
        "fliers",
        "means",
        "medians",
        "caps",
    ]:
        plt.setp(bp[element], color=edge_color)

    for patch in bp["boxes"]:
        patch.set_facecolor(fill_color)

In [ ]:
def add_training_data(ax, ctype, rf_pos, rf_etypes):
    """Add training-data boxplots."""

    rf_etypes_sorted = sort_rf_etypes(ctype, rf_etypes,)

    for pos, etype in zip(rf_pos, rf_etypes_sorted):
        add_boxplot(
            ax=ax,
            data=df_train[f"{ctype}_{etype}"],
            position=pos,
            edge_color="#000000",
            fill_color=colors[0],
        )

    return rf_etypes_sorted

In [ ]:
def prepare_literature_data(ctype, eco_zones):
    """Select, sort, and assign positions to literature data."""

    df_sub = df_lit.loc[
        df_lit["assigned_eco_zone"].isin(eco_zones)
        & df_lit["carbon_type"].eq(ctype)
    ].copy()
    
    df_sub = df_sub.sort_values(
        ["publication_year", "publication_author", "total_max",],
        ascending=[False, True, False,],
    )

    pos_map = {
        publication: pos
        for pos, publication in enumerate(
            df_sub["publication_id2"].unique(),
            start=1,
        )
    }

    df_sub["pos"] = (df_sub["publication_id2"].map(pos_map))

    return df_sub

In [ ]:
def add_literature_data(ax, df_sub):
    """Plot literature best estimates and ranges."""

    for marker, df_marker in df_sub.groupby("marker", sort=False,):
        for color, df_group in df_marker.groupby("color", sort=False,):
            # Best estimate
            ax.scatter(
                df_group["value_best"], 
                df_group["pos"], 
                marker=marker,
                c=color,
                zorder=3,
            )

            # Minimum
            ax.scatter(
                df_group["value_min"],
                df_group["pos"],
                marker=marker,
                c=color,
                zorder=3,
            )

            # Maximum
            ax.scatter(
                df_group["value_max"],
                df_group["pos"],
                marker=marker,
                c=color,
                zorder=3,
            )

            # Min-max range
            ax.hlines(
                y=df_group["pos"],
                xmin=df_group["value_min"],
                xmax=df_group["value_max"],
                color=color,
                zorder=2,
            )

In [ ]:
def add_publication_separators(ax, df_sub):
    """Add horizontal dashed lines between publications."""

    new_publication = (
        df_sub["publication_id"]
        .ne(df_sub["publication_id"].shift())
    )

    separator_pos = (
        df_sub.loc[new_publication, "pos"]
        .sub(0.5)
        .unique()
    )

    for pos in separator_pos:
        ax.axhline(
            pos,
            color="#000000",
            linestyle="--",
            linewidth=1,
            zorder=1,
        )

In [ ]:
def add_background_shading(ax, yticks):
    """Add alternating horizontal background shading."""

    y_min = int(np.floor(np.min(yticks)))
    y_max = int(np.ceil(np.max(yticks)))

    for y in range(y_min, y_max + 1, 2):
        ax.axhspan(y - 0.5, y + 0.5, color="black", alpha=0.05, zorder=0,)

In [ ]:
def format_subplot(ax, rf_pos, rf_etypes_sorted, df_sub, xlim_max,):
    """Format axes, labels, ticks, grid, and background."""

    # One literature label per y-position
    df_labels = (
        df_sub[["publication_id2", "pos", "assigned_name",]]
        .drop_duplicates("publication_id2")
    )
    yticks = [*rf_pos, *df_labels["pos"],]
    ytick_labels = [
        *[d_rf_labels[etype] for etype in rf_etypes_sorted],
        *df_labels["assigned_name"],
    ]
    ax.set_yticks(yticks, labels=ytick_labels,)
    ax.set_xlim(0, xlim_max,)
    ax.set_ylim(max(yticks) + 0.5, min(yticks) - 0.5,)
    ax.set_xlabel("tC ha$^{-1}$")
    ax.grid(axis="x", which="major", zorder=0,)
    add_background_shading(ax, yticks,)

In [ ]:
def plot_sub(ax, ctype, trop_temp, xlim_max):
    """Plot one tropical or temperate/boreal/global subplot."""

    settings = get_plot_settings(trop_temp)

    # Training data
    rf_etypes_sorted = add_training_data(
        ax=ax,
        ctype=ctype,
        rf_pos=settings["rf_pos"],
        rf_etypes=settings["rf_etype"],
    )

    # Literature data
    df_sub = prepare_literature_data(
        ctype=ctype,
        eco_zones=settings["eco_zones"],
    )

    add_literature_data(ax, df_sub)

    add_publication_separators(ax, df_sub)

    # Layout
    format_subplot(
        ax=ax,
        rf_pos=settings["rf_pos"],
        rf_etypes_sorted=rf_etypes_sorted,
        df_sub=df_sub,
        xlim_max=xlim_max,
    )

In [ ]:
def add_title(ax, text):
    """Add title to subplot."""

    ax.set_title(
        text,
        fontsize="x-large",
        fontweight="bold",
    )

In [ ]:
def add_legend(fig, ctype, ncol):
    """Add figure legend."""

    legend_specs = [
        ("Training data", "s", "#000000", colors[0],),
        ("Houghton et al. 2023", "s", dict_colors["houghton_2023"], dict_colors["houghton_2023"],),
        ("Goldstein et al. 2020", "s", dict_colors["goldstein_2020"], dict_colors["goldstein_2020"],),
        ("Mackey et al. 2020", "s", dict_colors["mackey_2020"], dict_colors["mackey_2020"], ),
        ("IPCC 2019", "s", dict_colors["ipcc_2019"], dict_colors["ipcc_2019"],),
        ("Hansis et al. 2015", "s", dict_colors["hansis_2015"], dict_colors["hansis_2015"],),
        ("Tropical", dict_marker["Tropical"], "#000000", "#000000",),
        ("Subtropical", dict_marker["Subtropical"], "#000000", "#000000",),
        ( "Temperate", dict_marker["Temperate"], "#000000", "#000000",),
        ("Boreal", dict_marker["Boreal"], "#000000", "#000000",),
        ("Global", dict_marker["Global"], "#000000", "#000000", ),
    ]

    if ctype == "csoil":
        exclude = {"Mackey et al. 2020", "IPCC 2019"}
        legend_specs = [item for item in legend_specs if item[0] not in exclude]

    handles = [
        Line2D(
            [0], [0],
            marker=marker,
            markeredgecolor=edge_color,
            markerfacecolor=face_color,
            lw=0,
        )
        for _, marker, edge_color, face_color
        in legend_specs
    ]

    labels = [label for label, *_ in legend_specs]

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=ncol,
        bbox_to_anchor=(0.5, -0.06),
        frameon=False,
    )

---

### Plot vegetation carbon

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 9))

# Tropical and subtropical
plot_sub(axs[0], ctype="cveg", trop_temp="trop", xlim_max=500,)
add_title(axs[0], "Tropics and Subtropics")

# Temperate, boreal, and global
plot_sub(axs[1], ctype="cveg", trop_temp="temp", xlim_max=500,)
add_title(axs[1], "Temperate, Boreal, and Global")

# Legend
add_legend(fig, ctype="cveg", ncol=6,)

# Layout
plt.tight_layout()

# Export
fstr_out = 'fig_m02_literature_cveg'
plt.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300,
            bbox_inches='tight', pad_inches=0.1)
plt.savefig(f'{dir08}png/{fstr_out}.png', dpi=300, 
            bbox_inches='tight', pad_inches=0.1)

---

### Plot soil carbon

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 9))

# Tropical and subtropical
plot_sub(axs[0], ctype="csoil", trop_temp="trop", xlim_max=800,)
add_title(axs[0], "Tropics and Subtropics")

# Temperate, boreal, and global
plot_sub(axs[1], ctype="csoil", trop_temp="temp", xlim_max=800,)
add_title(axs[1], "Temperate, Boreal, and Global")

# Legend
add_legend(fig, ctype="cveg", ncol=6,)

# Layout
plt.tight_layout()

# Export
fstr_out = 'fig_m04_literature_csoil'
plt.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300,
            bbox_inches='tight', pad_inches=0.1)
plt.savefig(f'{dir08}png/{fstr_out}.png', dpi=300, 
            bbox_inches='tight', pad_inches=0.1)